# NILM-Flow: Energy Disaggregation & Appliance Load Profiling
## REFIT Data Cleaning & Washing Machine Disaggregation Pipeline

**Author:** Kushagra Sharma  
**Dataset:** REFIT Electrical Load Measurements (Murray et al., 2015)  

---
### Notebook Roadmap
1. **Environment Setup & Data Ingestion**: Inspect dataset schema, units (Watts), and sampling rate (~8s).
2. **Raw Data Inspection & 1-Minute Cleaning (House 1)**: Audit timestamps, duplicates, outages, spikes, and resample to 1-minute.
3. **Exploratory Data Analysis of Washing Machines**: Analyze 5 households, implement physical cycle detection rule, examine diurnal & weekly profiles.
4. **Appliance Disaggregation Modeling**: Benchmark baselines vs XGBoost regressor using a strict chronological block split.
5. **Visualizations & Failure Mode Analysis**: Inspect actual vs. predicted profiles, feature importances, and overlapping load errors.

In [ ]:
# Setup environment and import modules
import sys, os
sys.path.insert(0, '..')
os.environ['PYTHONIOENCODING'] = 'utf-8'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import json

from src.config import RAW_DIR, CLEAN_DIR, PLOT_DIR, TABLE_DIR, SELECTED_HOUSES, WASHING_MACHINE_CHANNEL
from src.data_ingestion import load_raw_house, load_clean_house, data_summary

print('Environment and configuration loaded successfully!')

## Section 1: Raw Data Inspection & Quality Findings (House 1)
We examine the raw ~8-second polled readings across Part 1 and Part 2 for House 1.

In [ ]:
# Load raw inspection stats
with open('../outputs/tables/raw_house1_inspection.json', 'r') as f:
    raw_stats = json.load(f)

print(json.dumps(raw_stats, indent=2))

### Cleaning Pipeline Execution Log
We review the quantitative impact of deduplication (favoring Part 2 NaN semantics), spike clipping (>10 kW), short-gap interpolation (≤5 min), and 1-minute mean resampling.

In [ ]:
with open('../outputs/tables/house1_cleaning_log.json', 'r') as f:
    cleaning_log = json.load(f)

pd.DataFrame(list(cleaning_log.items()), columns=['Pipeline Metric', 'Value'])

In [ ]:
# Display before vs after 48-hour cleaning comparison plot
from IPython.display import Image
Image('../outputs/plots/house1_raw_vs_clean_48h.png')

## Section 2: Washing Machine Exploratory Data Analysis (>=4 Households)
We analyze 5 households (Houses 1, 2, 5, 7, 20) using our physical thresholding cycle detection rule:
* **Active Threshold**: $P \ge 15\text{ W}$
* **Bridge Pause**: Gaps $\le 5\text{ minutes}$ (soaking/drainage) merged
* **Minimum Duration**: $\ge 15\text{ minutes}$
* **Peak Power Validation**: Peak $\ge 500\text{ W}$ (confirms resistance water heating or spin extraction)

In [ ]:
# Cross-household summary table
summary_df = pd.read_csv('../outputs/tables/wm_cross_house_summary.csv')
summary_df

In [ ]:
# Representative 24-hour trace with detected cycle
Image('../outputs/plots/wm_representative_24h_trace.png')

In [ ]:
# High-resolution zoom of detected cycle signatures
Image('../outputs/plots/wm_detected_cycles_detail.png')

In [ ]:
# Cross-household behavioral comparisons (Diurnal, Weekly, Duration, Energy)
Image('../outputs/plots/wm_cross_house_comparison.png')

## Section 3: Appliance-Disaggregation Modeling (NILM)
We benchmarked our rolling-context XGBoost Regressor against an Always-Off baseline and a Diurnal Profile baseline on unseen test data using a strict contiguous chronological split (70% Train, 10% Validation, 20% Test).

In [ ]:
# Benchmark metrics table
metrics_df = pd.read_csv('../outputs/tables/nilm_model_metrics.csv')
metrics_df

In [ ]:
# Actual vs Predicted evaluation plot
Image('../outputs/plots/nilm_actual_vs_predicted_test.png')

In [ ]:
# Top feature importances
Image('../outputs/plots/nilm_feature_importance.png')

In [ ]:
# Model failure case analysis
Image('../outputs/plots/nilm_failure_cases.png')

## Section 4: Utility Implications & Recommendations
For executive summaries and utility-facing policy recommendations, please see [`Recommendation.md`](../Recommendation.md):
1. **SmartShift Laundry**: Dynamic ToD tariff shifting (08:00–13:00 peak) + 30°C cold wash nudges.
2. **Measurement & Verification**: Randomized Controlled Trial (RCT) evaluated via Difference-in-Differences (DiD) econometric regression.
3. **Domain Transfer Challenges**: Top-loading washers without heaters, high overlapping AC/geyser loads, and grid voltage swings in India.
4. **15-Min / 30-Min Resolution Degradation**: Mathematical demonstration of transient pulse smearing and the shift to Factorial HMMs / matrix factorization.